# Customer Retention Prediction — Exploratory Data Analysis

**Dataset:** IBM Telco Customer Churn  
**Goal:** Understand the structure of the data, identify patterns related to churn, and surface insights that will guide feature engineering and modelling decisions.

## 0. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# Allow imports from src/
sys.path.append(str(Path('..').resolve()))
from src.data_processing import download_data, load_data, clean_data

# Output directory for saved plots
PLOT_DIR = Path('../data/eda_plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Consistent aesthetics
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
CHURN_PALETTE = {0: '#4C72B0', 1: '#DD8452'}   # blue = retained, orange = churned

print('Setup complete. Plots will be saved to:', PLOT_DIR.resolve())

## 1. Load Raw Data

In [ ]:
download_data(path='../data/telco_churn.csv')
df_raw = load_data(path='../data/telco_churn.csv')

# Keep a cleaned copy for numeric analysis
df = clean_data(df_raw.copy())

print(f'Raw shape : {df_raw.shape}')
print(f'Clean shape: {df.shape}')

## 2. First Look — Shape, Head, Dtypes

In [ ]:
print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns\n')
df_raw.head()

In [ ]:
print('Column dtypes:')
print(df_raw.dtypes.to_string())

**Observations:**
- The dataset has 7,043 customers and 21 columns covering demographics, account info, and subscribed services.
- Most feature columns are `object` (categorical); only `tenure`, `MonthlyCharges`, and `SeniorCitizen` are numeric.
- `TotalCharges` is stored as `object` due to whitespace entries for new customers — needs conversion.
- `customerID` is a non-informative identifier and will be dropped during cleaning.

## 3. Missing Value Heatmap

In [ ]:
# Build a boolean missing-value matrix (rows = columns with any nulls, cols = samples)
# For the raw df, TotalCharges whitespace strings won't show as NaN yet;
# force-convert to surface them.
df_missing_check = df_raw.copy()
df_missing_check['TotalCharges'] = pd.to_numeric(
    df_missing_check['TotalCharges'], errors='coerce'
)

missing_counts = df_missing_check.isnull().sum()
cols_with_missing = missing_counts[missing_counts > 0].index.tolist()

fig, ax = plt.subplots(figsize=(12, 3))

if cols_with_missing:
    sns.heatmap(
        df_missing_check[cols_with_missing].isnull(),
        yticklabels=False,
        cbar=False,
        cmap='viridis',
        ax=ax,
    )
    ax.set_title('Missing Value Heatmap (yellow = missing)', fontsize=13)
    ax.set_xlabel('Column')
else:
    ax.text(0.5, 0.5, 'No missing values detected', ha='center', va='center',
            transform=ax.transAxes, fontsize=14)
    ax.set_title('Missing Value Heatmap', fontsize=13)

plt.tight_layout()
fig.savefig(PLOT_DIR / '01_missing_values.png', dpi=150)
plt.show()

print('Missing value counts (after TotalCharges coercion):')
print(missing_counts[missing_counts > 0] if cols_with_missing else 'None')

**Observations:**
- Only `TotalCharges` has missing values — 11 records where the customer is brand-new (tenure = 0), so the charge is an empty string rather than 0.
- These 11 rows represent <0.2% of the dataset; imputing with the column median is safe and has negligible impact on model performance.
- All other 20 columns are complete — no systemic data-quality issues.

## 4. Churn Distribution

In [ ]:
churn_counts = df['Churn'].value_counts().rename({0: 'Retained', 1: 'Churned'})
churn_pct = (churn_counts / churn_counts.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
bars = axes[0].bar(
    churn_counts.index,
    churn_counts.values,
    color=['#4C72B0', '#DD8452'],
    edgecolor='white',
    width=0.5,
)
for bar, pct in zip(bars, churn_pct.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 40,
        f'{pct}%',
        ha='center',
        fontsize=12,
        fontweight='bold',
    )
axes[0].set_title('Churn Distribution (count)', fontsize=13)
axes[0].set_ylabel('Number of Customers')
axes[0].set_xlabel('')

# Pie chart
axes[1].pie(
    churn_counts.values,
    labels=churn_counts.index,
    autopct='%1.1f%%',
    colors=['#4C72B0', '#DD8452'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
axes[1].set_title('Churn Distribution (proportion)', fontsize=13)

plt.suptitle('Customer Churn vs. Retention', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(PLOT_DIR / '02_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(churn_counts.to_string())

**Observations:**
- The dataset is **class-imbalanced**: ~73% retained vs ~27% churned.
- This ~3:1 imbalance means accuracy alone is a misleading metric — models should be evaluated with AUC-ROC, F1-score, and precision-recall curves.
- Techniques such as class weighting (`scale_pos_weight` in XGBoost) or SMOTE oversampling should be considered during training to prevent the model from being biased toward predicting "No Churn".

## 5. Correlation Heatmap (Numerical Features)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))   # show lower triangle only

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    square=True,
    ax=ax,
)
ax.set_title('Correlation Heatmap — Numerical Features', fontsize=13)
plt.tight_layout()
fig.savefig(PLOT_DIR / '03_correlation_heatmap.png', dpi=150)
plt.show()

**Observations:**
- `tenure` and `TotalCharges` are highly correlated (~0.83) — long-tenured customers accumulate more charges naturally. We should be mindful of multicollinearity when interpreting coefficients in linear models (less of a concern for tree-based models).
- `MonthlyCharges` and `TotalCharges` are moderately correlated (~0.65).
- `Churn` has the strongest *negative* correlation with `tenure`: customers who leave tend to do so early in their lifecycle.
- `MonthlyCharges` shows a positive correlation with churn, suggesting higher-bill customers are more likely to leave.

## 6. Distribution Plots Grouped by Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

dist_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
titles = ['Tenure (months)', 'Monthly Charges ($)', 'Total Charges ($)']

for ax, feat, title in zip(axes, dist_features, titles):
    for label, color, name in zip([0, 1], ['#4C72B0', '#DD8452'], ['Retained', 'Churned']):
        subset = df.loc[df['Churn'] == label, feat]
        subset.plot.kde(ax=ax, color=color, label=name, linewidth=2)
        ax.axvline(subset.median(), color=color, linestyle='--', linewidth=1, alpha=0.7)

    ax.set_title(f'Distribution of {title}\nby Churn Status', fontsize=11)
    ax.set_xlabel(title)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Feature Distributions: Retained vs. Churned Customers',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(PLOT_DIR / '04_distributions_by_churn.png', dpi=150, bbox_inches='tight')
plt.show()

**Observations:**

**Tenure:**  
Churned customers cluster heavily at low tenure values (0–12 months). The retained group's distribution is much flatter and spread across the full range, indicating that surviving the first year drastically increases the likelihood of long-term retention. **Tenure is likely the single strongest predictor of churn.**

**Monthly Charges:**  
Churned customers are concentrated at higher monthly charges (\$60–\$100+). Retained customers peak at the lower end (~\$20). This suggests that premium plan subscribers are at higher churn risk — possibly due to perceived value mismatches.

**Total Charges:**  
Churned customers have lower total charges (they leave before accumulating spend). Retained customers show a long right tail reflecting years of accumulated billing. This distribution is a downstream consequence of tenure rather than an independent signal.

## 7. Box Plots — Numerical Features by Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feat, title in zip(axes, dist_features, titles):
    sns.boxplot(
        data=df,
        x='Churn',
        y=feat,
        palette={0: '#4C72B0', 1: '#DD8452'},
        width=0.45,
        flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.4},
        ax=ax,
    )
    ax.set_xticklabels(['Retained', 'Churned'])
    ax.set_title(f'{title} by Churn', fontsize=11)
    ax.set_xlabel('')

plt.suptitle('Box Plots: Retained vs. Churned', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(PLOT_DIR / '05_boxplots_by_churn.png', dpi=150, bbox_inches='tight')
plt.show()

**Observations:**
- The median tenure for churned customers (~10 months) is dramatically lower than for retained ones (~38 months), confirming tenure as a top-tier predictor with clear separation.
- Monthly charges for churned customers are higher on average, with less variance — churners are concentrated in the premium billing tier.
- Total charges outliers in the retained group represent long-tenure, high-spend "loyal" customers — a potentially valuable segment to protect.

## 8. Summary Statistics

In [ ]:
total_records = len(df)
churn_rate = df['Churn'].mean() * 100
n_features = df.shape[1] - 1   # exclude target

print('=' * 50)
print('EDA SUMMARY')
print('=' * 50)
print(f'  Total records  : {total_records:,}')
print(f'  Churn rate     : {churn_rate:.2f}%')
print(f'  Number of features (raw, excl. ID): {n_features}')
print(f'  Plots saved to : {PLOT_DIR.resolve()}')
print('=' * 50)

print('\nNumerical feature statistics by Churn:')
df.groupby('Churn')[dist_features].median().rename(index={0: 'Retained', 1: 'Churned'})

## 9. Key Takeaways for Modelling

| Insight | Implication |
|---|---|
| Class imbalance (~73/27) | Use AUC-ROC + F1; apply `scale_pos_weight` in XGBoost |
| `tenure` is the strongest separator | Will rank high in feature importance; consider interaction features (e.g. tenure × MonthlyCharges) |
| `TotalCharges` ≈ `tenure × MonthlyCharges` | High multicollinearity; may drop `TotalCharges` or use PCA |
| Churners skew to high `MonthlyCharges` | Segment analysis by contract type and internet service will be informative |
| Only 11 missing values (TotalCharges) | Median imputation is safe; no complex imputation strategy needed |